In [5]:
import os
import pandas as pd
from pathlib import Path

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\GBIF')

# Prepare for merging

In [6]:
merged_GBIF = pd.read_parquet("merged_GBIF.parquet")
column_coverage_GBIF = pd.DataFrame({
    "column": merged_GBIF.columns,
    "non_null_count": merged_GBIF.notna().sum().values,
    "total_rows": len(merged_GBIF),
})

column_coverage_GBIF["coverage_pct"] = (
    column_coverage_GBIF["non_null_count"] / column_coverage_GBIF["total_rows"] * 100
)

column_coverage_GBIF = column_coverage_GBIF.sort_values(
    by=["coverage_pct", "non_null_count"], ascending=False
)

column_coverage_GBIF.to_csv("column_coverage_GBIF.csv", index=False)

column_coverage_GBIF.head()

,column,non_null_count,total_rows,coverage_pct
0,gbifID,2746900,2746900,100.0
1,accessRights,2746900,2746900,100.0
2,bibliographicCitation,2746900,2746900,100.0
3,language,2746900,2746900,100.0
4,license,2746900,2746900,100.0


In [7]:
gbif = pd.read_parquet("merged_GBIF.parquet")

dtype_spec = pd.read_excel(Path("..") / "column_data_types.xlsx", sheet_name=0)
dtype_map = dict(zip(dtype_spec["column"], dtype_spec["final"]))

column_map = {
    "sourceID": "gbifID",
    "verbatimID": "occurrenceID",
    "datasetKey": "datasetKey",
    "issue": "issue",
    "decimalLatitudeStart": "decimalLatitude",
    "decimalLongitudeStart": "decimalLongitude",
    "footprintWKT": "footprintWKT",
    "coordinateUncertaintyInMeters": "coordinateUncertaintyInMeters",
    "hasGeospatialIssues": "hasGeospatialIssues",
    "depthA_m": "depth",
    "eventID": "eventID",
    "parentEventID": "parentEventID",
    "waterBody": "waterBody",
    "verbatimDate": "eventDate",
    "startDayOfYear": "startDayOfYear",
    "endDayOfYear": "endDayOfYear",
    "year": "year",
    "eventRemarks": "eventRemarks",
    "verbatimScientificName": "verbatimScientificName",
    "acceptedScientificName": "acceptedScientificName",
    "identificationQualifier": "identificationQualifier",
    "lifeStage": "lifeStage",
    "sex": "sex",
    "taxonomicStatus": "taxonomicStatus",
    "taxonomicIssue": "taxonomicIssue",
    "taxonRank": "taxonRank",
    "phylum": "phylum",
    "class": "class",
    "order": "order",
    "family": "family",
    "genus": "genus",
    "individualCount": "individualCount",
    "basisOfRecord": "basisOfRecord",
    "gear": "samplingProtocol",
    "acceptedTaxonKey": "acceptedTaxonKey"
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeospatialIssues",
    "depthA_m",
    "depthAType",
    "depthB_m",
    "depthBType",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "verbatimDate",
    "startDayOfYear",
    "endDayOfYear",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "areaSampled",
    "areaSampledUnit",
    "volumeSampled_m3",
    "samplingEffort",
    "trawlingDistance_m",
    "biomass",
    "biomassUnit",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "meshSize_mm",
    "sieveFraction_µm",
    "flagged",
    "flaggedComment",
    "acceptedTaxonKey"
]


def normalize_dtype(dtype_name):
    dtype_name = str(dtype_name).strip().lower()
    return {
        "int32": "Int32",
        "float32": "Float32",
        "float64": "Float64",
        "boolean": "boolean",
        "string": "string",
    }.get(dtype_name, dtype_name)


def empty_series(index, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name in {"Int32", "Float32", "Float64", "boolean", "string"}:
        return pd.Series(pd.NA, index=index, dtype=dtype_name)
    return pd.Series(pd.NA, index=index, dtype="string")


def coerce_series(series, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name == "boolean":
        # Handle string boolean representations (true/false, True/False) and numeric 0/1
        def to_bool(x):
            if pd.isna(x):
                return pd.NA
            str_val = str(x).strip().lower()
            if str_val in ("true", "1", "1.0"):
                return True
            elif str_val in ("false", "0", "0.0"):
                return False
            else:
                return pd.NA
        return pd.Series([to_bool(x) for x in series], index=series.index, dtype="boolean")
    if dtype_name in {"Int32", "Float32", "Float64"}:
        return pd.to_numeric(series, errors="coerce").astype(dtype_name)
    return series.astype("string")


gbif_final = pd.DataFrame(index=gbif.index)
missing_sources = []

for col in output_columns:
    target_dtype = dtype_map.get(col, "string")
    if col in column_map:
        src = column_map[col]
        if src in gbif.columns:
            gbif_final[col] = coerce_series(gbif[src], target_dtype)
        else:
            gbif_final[col] = empty_series(gbif.index, target_dtype)
            missing_sources.append(src)
    else:
        gbif_final[col] = empty_series(gbif.index, target_dtype)

gbif_final = gbif_final[output_columns]

# Add sourceDatabase column with value 'GBIF'
gbif_final["sourceDatabase"] = "GBIF"

gbif_final.to_parquet("gbif_final.parquet", index=False)

print(f"Saved gbif_final.parquet with {len(gbif_final):,} rows and {len(gbif_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in gbif:", sorted(set(missing_sources)))

# Show which columns are boolean type
boolean_columns = [col for col in output_columns if dtype_map.get(col, "string") == "boolean"]
print(f"\nBoolean columns: {boolean_columns}")
print("Added sourceDatabase column with value 'GBIF'.")

Saved gbif_final.parquet with 2,746,900 rows and 63 columns.

Boolean columns: ['hasGeospatialIssues', 'flagged']
Added sourceDatabase column with value 'GBIF'.


# Check column data types

In [8]:
gbif_final = pd.read_parquet("gbif_final.parquet")

# Column type overview
column_types = pd.DataFrame({
    "column": gbif_final.columns,
    "dtype": gbif_final.dtypes.astype(str).values,
    "non_null_count": gbif_final.notna().sum().values,
    "null_count": gbif_final.isna().sum().values,
    "non_null_pct": (gbif_final.notna().sum().values / len(gbif_final)) * 100,
})

column_types = column_types.sort_values(
    by=["non_null_count", "column"], ascending=[False, True]
 )

column_types.to_csv("gbif_final_column_types.csv", index=False)

column_types.head(20)

,column,dtype,non_null_count,null_count,non_null_pct
30,acceptedScientificName,string,2746900,0,100.0
62,acceptedTaxonKey,string,2746900,0,100.0
54,basisOfRecord,string,2746900,0,100.0
38,class,string,2746900,0,100.0
4,datasetKey,string,2746900,0,100.0
7,decimalLatitudeStart,Float64,2746900,0,100.0
8,decimalLongitudeStart,Float64,2746900,0,100.0
19,eventID,string,2746900,0,100.0
28,eventRemarks,string,2746900,0,100.0
40,family,string,2746900,0,100.0
